# PID Control


<span id="실행-준비"></span>

## Execution setup

The Python option uses python-control. If the module is missing, run `%pip install control==0.10.2` in a Colab or Jupyter code cell.

<span id="학습-목표"></span>

## Learning objectives

Selecting a gain based only on response speed can leave the system vulnerable to overshoot or model changes. Compare different performance metrics.

<span id="ppipid-기준-비교"></span>

## Baseline P, PI, and PID comparison

<span id="시작-조건"></span>

### Starting conditions

Motor parameters: $J=.01$,$b=.1$,$K=.01$,$R=1$,$L=.5$. P=(500,0,0), PI=(500,300,0), PID=(500,300,30).

<span id="진행-방법"></span>

### Procedure

1.  Build three closed-loop models using negative unity feedback.
2.  Plot the unit-step responses over $0\le t\le 2$ s in the same figure, and check the poles first.
3.  Record overshoot, settling time, and steady-state error for stable models.

<span id="남길-결과"></span>

### Results to retain

- A table with 3 controller rows, response plots, and a brief comparison of their roles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
G=.01/((.01*s+.1)*(.5*s+1)+.01**2)
configs={'P':(500,0,0),'PI':(500,300,0),'PID':(500,300,30)}
for name,(kp,ki,kd) in configs.items():
    C=kp+ki/s+kd*s
    T=ctl.feedback(C*G,1)
    print(name,ctl.poles(T))
# TODO: Record responses and performance metrics.


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**Controller:**

**Kp:**

**Ki:**

**Kd:**

**Stability:**

**Overshoot \[%\]:**

**Settling Time \[s\]:**

**Steady-State Error:**

<span id="확인할-점"></span>

### Points to check

- Are the plant, input, and time axis the same, with only the controller changing?
- For unstable or unconverged cases, did you give a reason instead of reporting metrics?

<span id="한-항씩-변화시키기"></span>

## Changing one term at a time

<span id="시작-조건-1"></span>

### Starting conditions

Starting from the PID baseline (500,300,30), change only Kp to 250,500,750. Keep Ki and Kd fixed.

<span id="진행-방법-1"></span>

### Procedure

1.  Use the PID result from Activity 1, $(500,300,30)$, as the baseline, and change only $K_p$ to 250 and 750 in turn. Keep $K_i=300$, $K_d=30$, the plant, and the time axis unchanged.
2.  Plot both unit-step responses alongside the baseline curve in the same figure. Use the same `ctl.step_info` settings to calculate overshoot, settling time, and steady-state error, and record them in the same columns as Activity 1.
3.  Choose one of the two additional settings. Use the table values to explain which metric you aim to improve over the baseline and which changes you are willing to accept.

<span id="남길-결과-1"></span>

### Results to retain

- Response plots and a performance table for both settings. One sentence explaining your choice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
G=.01/((.01*s+.1)*(.5*s+1)+.01**2)
for kp in [250,500,750]:
    T=ctl.feedback((kp+300/s+30*s)*G,1)
    print(kp,ctl.dcgain(T))
# TODO: Complete the time responses and comparison of the second gain.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**Changed term:**

**Value:**

**Overshoot \[%\]:**

**Settling Time \[s\]:**

**Steady-State Error:**

<span id="확인할-점-1"></span>

### Points to check

- Did you change only one gain at a time?
- Did you support claims of faster response with specific metrics?

<span id="입력-한계와-적분-누적"></span>

## Input limits and integrator windup

<span id="시작-조건-2"></span>

### Starting conditions

1st-order plant $\dot y=-y+u$, reference 2→.5 (changes at 5 s), $u\in[0,1]$, `dt=.01` s. Use $K_p=2$, $K_i=1$.

<span id="진행-방법-2"></span>

### Procedure

1.  In the supplied PI simulation, plot the actual input alongside the input before limiting.
2.  Compare conditional integration enabled and disabled.
3.  Explain why the error persists after lowering the reference, relating it to the integrator state.

<span id="남길-결과-2"></span>

### Results to retain

- Three plots of output, input, and integrator state, and a comparison table.

In the comparison table, IAE is $\int_{5}^{t_{\mathrm{last}}}|r(t)-y(t)|\,dt$ from the reference-change time to the final recording time. Calculate input saturation duration by multiplying the number of samples where the limited $u$ is 0 or 1 over the entire record by `dt`.

Use the same interval and calculation method for both integration approaches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def simulate(antiwindup):
    t=np.arange(0,12,.01);y=0.;integral=0.;rows=[]
    for now in t:
        ref=2. if now<5 else .5;e=ref-y;raw=2*e+integral;u=np.clip(raw,0,1)
        if not antiwindup or (0<raw<1) or (raw>=1 and e<0) or (raw<=0 and e>0):integral+=e*.01
        y+=(-y+u)*.01;rows.append((ref,y,raw,u,integral))
    return t,np.array(rows)
for flag in [False,True]:
    time,trace=simulate(flag);plt.plot(time,trace[:,1],label=str(flag))
plt.xlabel('Time [s]');plt.ylabel('Output');plt.legend()
# TODO: Add the input, integrator state, and metrics after 5 s.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Integration handling:**

**IAE after the reference change:**

**Maximum integral value:**

**Input saturation duration \[s\]:**

<span id="확인할-점-2"></span>

### Points to check

- Did you distinguish input values before and after limiting?
- Did you compare only integration handling without also changing the PI gains?